# Phase 3: Data Preparation
**CRISP-DM Purpose:** Transform raw data into modeling-ready form. All cleaning, feature engineering, and splitting decisions are made and documented here.

---

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

from src.config import SEED, TEST_SIZE, TARGET
from src.data_io import load_all_csvs
from src.features import build_feature_matrix, add_label
from src.modeling import build_preprocessor

from sklearn.model_selection import train_test_split

In [11]:
tables       = load_all_csvs()
residents    = tables['residents']
health_df    = tables['health']
edu_df       = tables['education']
sessions_df  = tables['sessions']
visits_df    = tables['visitations']
plans_df     = tables['plans']
incidents_df = tables['incidents']

print(f'Loaded: {len(residents)} residents')

Loaded: 60 residents


## 3.1 Feature Engineering

In [12]:
# Build joined feature set and label (logic lives in src/features.py)
df = build_feature_matrix(residents, health_df, edu_df, sessions_df, visits_df, plans_df, incidents_df)
df = add_label(df)

# Training scope: residents with known outcomes only.
# 'In Progress' residents have NaN labels — excluded from training, scored at inference.
# Completed=1, Not Started=0, On Hold=0 form the ground truth.
df_model = df[df[TARGET].notna()].copy().reset_index(drop=True)
df_model[TARGET] = df_model[TARGET].astype(int)

print(f'All residents: {len(df)}')
print(f'Training scope (known outcomes): {len(df_model)}')
print(f'Held for inference (In Progress): {df[TARGET].isna().sum()}')
print(f'\n{TARGET} distribution:')
print(df_model[TARGET].value_counts().rename({1: 'Reintegration Ready (1)', 0: 'Not Ready (0)'}))

All residents: 60
Training scope (known outcomes): 39
Held for inference (In Progress): 21

reintegration_ready distribution:
reintegration_ready
Not Ready (0)              20
Reintegration Ready (1)    19
Name: count, dtype: int64


## 3.2 Column Inclusion / Exclusion Report

In [13]:
# Columns excluded and the reason why
EXCLUDE = {
    # --- Identifiers ---
    'resident_id':                'Identifier — no predictive signal',
    'case_control_no':            'Identifier — no predictive signal',
    'internal_code':              'Identifier — no predictive signal',
    'safehouse_id':               'Identifier — facility code, not a resident attribute',

    # --- PII / free-text / high-cardinality ---
    'place_of_birth':             'PII — high-cardinality free text',
    'religion':                   'Sensitive demographic — not predictive of outcome',
    'pwd_type':                   'PII — sparse free text; predictive signal captured by is_pwd flag',
    'special_needs_diagnosis':    'PII — free-text clinical notes',
    'referring_agency_person':    'PII — name of referrer',
    'assigned_social_worker':     'PII — worker names; not a resident attribute',
    'initial_case_assessment':    'PII / free-text notes — not structured for modeling',
    'notes_restricted':           'Restricted free-text notes',

    # --- Raw dates (already encoded as numeric features) ---
    'date_of_birth':              'Encoded via age_upon_admission',
    'date_of_admission':          'Encoded via length_of_stay_days',
    'date_colb_registered':       'Raw date — sparse; not encoded as duration',
    'date_colb_obtained':         'Raw date — sparse; not encoded as duration',
    'date_case_study_prepared':   'Raw date — administrative; not predictive',
    'date_enrolled':              'Raw date — encoded via length_of_stay_days',
    'created_at':                 'Raw timestamp — administrative',

    # --- Leakage ---
    'reintegration_status':       '⚠ LEAKAGE — directly encodes the label (Completed → 1)',
    'date_closed':                '⚠ LEAKAGE — only populated for Completed cases; reveals label',

    # --- Redundant: risk (encoded as risk_delta + current_risk_numeric) ---
    'initial_risk_level':         'Encoded as initial_risk_numeric',
    'current_risk_level':         'Encoded as current_risk_numeric',
    'initial_risk_numeric':       'Redundant — direction captured by risk_delta; level by current_risk_numeric',
    'risk_improvement':           'Redundant binary of risk_delta > 0 — captured by risk_delta',

    # --- Redundant: raw strings already encoded as numeric ---
    'length_of_stay':             'Encoded as length_of_stay_days',

    # --- Redundant age (correlated with age_upon_admission + length_of_stay_days) ---
    'present_age':                'Redundant — age_upon_admission + length_of_stay_days already encode this',

    # --- Redundant abuse sub-categories (summed into abuse_complexity_score) ---
    'sub_cat_physical_abuse':     'Redundant — component of abuse_complexity_score',
    'sub_cat_sexual_abuse':       'Redundant — component of abuse_complexity_score',
    'sub_cat_osaec':              'Redundant — component of abuse_complexity_score',
    'sub_cat_trafficked':         'Redundant — component of abuse_complexity_score',
    'sub_cat_child_labor':        'Redundant — component of abuse_complexity_score',

    # --- Redundant family vulnerability flags (summed into family_vulnerability_score) ---
    'family_is_4ps':              'Redundant — component of family_vulnerability_score',
    'family_solo_parent':         'Redundant — component of family_vulnerability_score',
    'family_indigenous':          'Redundant — component of family_vulnerability_score',
    'family_parent_pwd':          'Redundant — component of family_vulnerability_score',
    'family_informal_settler':    'Redundant — component of family_vulnerability_score',

    # --- Redundant plan counts (captured by plan_achieved_ratio and plan_blocked_ratio) ---
    'plan_n_achieved':            'Redundant — captured by plan_achieved_ratio',
    'plan_n_on_hold':             'Redundant — captured by plan_blocked_ratio',

    # --- Target ---
    TARGET:                       'Target column',
}

print('Excluded columns and reasons:')
for col, reason in EXCLUDE.items():
    print(f'  {col:<35} → {reason}')

feature_cols = [c for c in df_model.columns if c not in EXCLUDE]
print(f'\nFeature columns ({len(feature_cols)} total):')
print(feature_cols)

Excluded columns and reasons:
  resident_id                         → Identifier — no predictive signal
  case_control_no                     → Identifier — no predictive signal
  internal_code                       → Identifier — no predictive signal
  safehouse_id                        → Identifier — facility code, not a resident attribute
  place_of_birth                      → PII — high-cardinality free text
  religion                            → Sensitive demographic — not predictive of outcome
  pwd_type                            → PII — sparse free text; predictive signal captured by is_pwd flag
  special_needs_diagnosis             → PII — free-text clinical notes
  referring_agency_person             → PII — name of referrer
  assigned_social_worker              → PII — worker names; not a resident attribute
  initial_case_assessment             → PII / free-text notes — not structured for modeling
  notes_restricted                    → Restricted free-text notes
  date_o

In [14]:
X = df_model[feature_cols]
y = df_model[TARGET]

numeric_cols     = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

print(f'Numeric features     ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

Numeric features     (46): ['sub_cat_orphaned', 'sub_cat_cicl', 'sub_cat_at_risk', 'sub_cat_street_child', 'sub_cat_child_with_hiv', 'is_pwd', 'has_special_needs', 'age_upon_admission', 'current_risk_numeric', 'risk_delta', 'length_of_stay_days', 'abuse_complexity_score', 'family_vulnerability_score', 'hlth_n_records', 'hlth_avg_general', 'hlth_avg_nutrition', 'hlth_avg_sleep', 'hlth_avg_energy', 'hlth_pct_medical', 'hlth_pct_dental', 'hlth_pct_psychological', 'hlth_trend', 'edu_n_records', 'edu_avg_attendance', 'edu_avg_progress', 'edu_latest_completion', 'edu_trend', 'sess_n_sessions', 'sess_pct_progress_noted', 'sess_pct_concerns_flagged', 'sess_pct_referral_made', 'sess_avg_duration_min', 'sess_pct_positive_end', 'sess_emotional_trend', 'vis_n_visits', 'vis_n_reintegration_assessments', 'vis_pct_favorable', 'vis_avg_cooperation', 'vis_pct_safety_concerns', 'plan_n_plans', 'plan_achieved_ratio', 'plan_blocked_ratio', 'inc_total_incidents', 'inc_high_severity_count', 'inc_pct_unresol

## 3.3 Train / Test Split

Test set is **frozen here and touched exactly once** — in Phase 5 final evaluation.  
All cross-validation and hyperparameter tuning in Phase 4 uses `X_train` / `y_train` only.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,   # preserves class ratio in both splits
)

print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')
print(f'\nTrain {TARGET} rate: {y_train.mean():.1%}')
print(f'Test  {TARGET} rate: {y_test.mean():.1%}')

assert abs(y_train.mean() - y_test.mean()) < 0.05, 'Stratification failed — class rates diverged'
print('\n✓ Stratification verified — class rates match within 5%')

Train: 31 rows  |  Test: 8 rows

Train reintegration_ready rate: 48.4%
Test  reintegration_ready rate: 50.0%

✓ Stratification verified — class rates match within 5%


## 3.4 Preprocessing Pipeline

All imputation and scaling is fitted **inside a `sklearn.Pipeline`** — never on the full dataset before splitting.  
This prevents validation statistics from leaking into training.

| Column type | Imputation | Scaling |
|---|---|---|
| Numeric | Median (robust to skew and outliers) | StandardScaler |
| Categorical | Most frequent | OneHotEncoder (handle_unknown='ignore') |

In [16]:
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

# Dry-run fit on train to inspect output shape
X_train_transformed = preprocessor.fit_transform(X_train)
print(f'Preprocessor output shape: {X_train_transformed.shape}')
print(f'  → {len(numeric_cols)} numeric + {X_train_transformed.shape[1] - len(numeric_cols)} encoded categorical columns')

Preprocessor output shape: (31, 66)
  → 46 numeric + 20 encoded categorical columns


## 3.5 Imbalance Check

In [17]:
counts = y_train.value_counts()
ratio  = counts.max() / counts.min()

print(f'Training class counts:\n{counts.rename({0: "Not Ready", 1: "Reintegration Ready"})}')
print(f'\nImbalance ratio: {ratio:.1f}:1')

if ratio >= 4:
    print('⚠ Imbalance ≥ 4:1 → will use class_weight="balanced" in Phase 4 models.')
    print('  Will report per-class precision/recall (not just overall accuracy).')
    print('  Will include PR curve alongside ROC curve in Phase 5.')
else:
    print('✓ Imbalance < 4:1 — standard training acceptable, but class_weight="balanced" applied as precaution.')

Training class counts:
reintegration_ready
Not Ready              16
Reintegration Ready    15
Name: count, dtype: int64

Imbalance ratio: 1.1:1
✓ Imbalance < 4:1 — standard training acceptable, but class_weight="balanced" applied as precaution.


## 3.6 Save Processed Splits

In [18]:
import json
from src.config import DATA_PROCESSED

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

X_train.assign(**{TARGET: y_train}).to_csv(DATA_PROCESSED / 'train.csv', index=False)
X_test.assign(**{TARGET: y_test}).to_csv(DATA_PROCESSED / 'test.csv', index=False)

with open(DATA_PROCESSED / 'feature_cols.json', 'w') as f:
    json.dump({
        'feature_cols':     feature_cols,
        'numeric_cols':     numeric_cols,
        'categorical_cols': categorical_cols,
    }, f, indent=2)

print(f'Saved to {DATA_PROCESSED}:')
print(f'  train.csv          ({len(X_train)} rows)')
print(f'  test.csv           ({len(X_test)} rows)')
print(f'  feature_cols.json')

Saved to C:\Users\apier\OneDrive\Documents\Code\Intex-II\intex-w26\pipeline\resident_reintegration\data\processed:
  train.csv          (31 rows)
  test.csv           (8 rows)
  feature_cols.json


## 3.7 Preparation Summary

| Decision | Choice | Reason |
|---|---|---|
| Training scope | Known-outcome residents (Completed / Not Started / On Hold) | In Progress residents have no ground truth — scored at inference |
| Label | `reintegration_ready` = 1 if `reintegration_status == 'Completed'` | True confirmed outcome, not a proxy threshold |
| `reintegration_status` excluded | Yes | Directly encodes the label — leakage |
| `date_closed` excluded | Yes | Only populated for Completed cases — leakage |
| PII / free-text excluded | Yes | Not predictive; privacy |
| Raw date columns excluded | Yes | Encoded as numeric duration features |
| Redundant string columns excluded | Yes | `initial/current_risk_level`, `length_of_stay` already encoded numerically |
| Numeric imputation | Median | Robust to right-skewed and zero-inflated count features |
| Categorical imputation | Most frequent | Safe default; few missings expected |
| Encoding | OneHotEncoder (handle_unknown='ignore') | Handles unseen categories at inference |
| Split | 80/20 stratified | Preserves class ratio; test set frozen |
| Imbalance handling | `class_weight='balanced'` in model | Applied in Phase 4 |
| Inference scope | In Progress residents only | We score residents still in the program |

---
**Proceed to Phase 4: Modeling →** `04_modeling.ipynb`